In [2]:
!pip install -q transformers datasets torchaudio

In [4]:
from huggingface_hub import login
login()

In [15]:
import torch
import numpy as np
from transformers import AutoModelForCTC, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

model = AutoModelForCTC.from_pretrained(model_id)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    device=device,
)

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [16]:
from datasets import load_dataset, Audio
dataset = load_dataset("linagora/linto-dataset-audio-ar-tn", trust_remote_code=True)

test_dataset = dataset["test"]


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'linagora/linto-dataset-audio-ar-tn' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'linagora/linto-dataset-audio-ar-tn' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

In [17]:
from datasets import Audio
test_dataset = dataset["test"].cast_column("audio", Audio(sampling_rate=16000))

In [18]:
import re
chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\‘\’\“\”]'

def remove_special_characters(batch):
    batch["transcript"] = re.sub(chars_to_ignore_regex, '', batch["transcript"]).lower()
    return batch
test_dataset = test_dataset.map(remove_special_characters)

<>:2: SyntaxWarning: invalid escape sequence '\,'
<>:2: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_21294/2941018406.py:2: SyntaxWarning: invalid escape sequence '\,'
  chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\‘\’\“\”]'


In [19]:
from datasets import Dataset

def explode_segments(dataset):
    rows = []

    for row in dataset:
        audio = row["audio"]

        for seg in row["segments"]:
            start = int(seg["start"] * 16000)
            end = int(seg["end"] * 16000)

            segment_audio = audio["array"][start:end]

            rows.append({
                "audio": {
                    "array": segment_audio,
                    "sampling_rate": 16000
                },
                "transcript": seg["transcript"]
            })

    return Dataset.from_list(rows)
segmented_test = explode_segments(test_dataset)

In [20]:
!pip install -q evaluate jiwer

In [21]:
import evaluate
from tqdm.auto import tqdm
import numpy as np


wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def data_generator():
    for x in segmented_test:
        yield {
    "array": np.array(x["audio"]["array"]),
    "sampling_rate": 16000
}

references = segmented_test["transcript"]
predictions = []

print(f"Running inference on {len(segmented_test)} samples...")
for out in tqdm(pipe(data_generator(), batch_size=4), total=len(segmented_test)):
    predictions.append(out["text"])

predictions = [remove_special_characters({"transcript": p})["transcript"] for p in predictions]
# Calculate metrics
final_wer = wer_metric.compute(predictions=predictions, references=references)
final_cer = cer_metric.compute(predictions=predictions, references=references)

print(f"WER: {final_wer:.4f}")
print(f"CER: {final_cer:.4f}")

Running inference on 1013 samples...


  0%|          | 0/1013 [00:00<?, ?it/s]

WER: 0.9148
CER: 0.5187


In [22]:
import pandas as pd
results_df = pd.DataFrame({"Reference": references, "Prediction": predictions})
display(results_df.head(50))

,Reference,Prediction
0,ويرحمني و إياكم في الدنيا و الآخرة,ويرحَمْن وإيكُمُ الدّنْيَا وَالآخرة
1,إشك أناهي الحكومة إلي باش تدخل لك في اصلاحات خ...,اشكئ اناي الحكومة البشتت خلك في إصلحات خطيرة
2,وهو قاعد ستة شهر,ووَقَعْتْ سَتَشُنُّ
3,إذن نحن بحاجة إلى مجلس تأسيسي إن ينجم ينظم كل هذا,إذا نحن بحاجة إلى مجل التأسيس لماء نج م نظم كل...
4,الإدارة العامة للميترو طالبة خدامة و الإدارة ا...,سبدر العامل المتْرُوتاب خدَم واقدل العامطبل كل...
5,صندوق عجب صباح الخير بربي حبيت نتكلم بربي شوفو...,سقعج بسبعخية برضي حبتن كلم برض شفوالنا الأسوام...
6,كيفاش تعملت قوانينو آش بيها المنظومة هذيكة متا...,كيف اشت عمْل القاونينَ اشباها المنظُومة ذكرط ع...
7,أقعد معايا سليم خلي ناخذوا جمال معانا بالتليفو...,فما يستمخلن خذ جميل معنب التلفون جمل ولعذّما
8,معانا محرز بالتليفون مرحبا,معنا محرز بالتلف مرحبك
9,كيفاش ينجم يكون القرار هذايا ناجح ويوصل للإقلا...,كيف شنجمل إيكون القرار هداي أن يجح و وصل للإق...


In [24]:
results_df = pd.DataFrame({
    "Reference": references,
    "Prediction": predictions
})

results_df.to_csv("whisper_large_v3_turbo_results.csv", index=False)